In [69]:
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
from functools import partial
import random


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm.auto import tqdm


import joblib
import pickle


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, confusion_matrix
)


from scipy import sparse


try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
except Exception:
    MultilabelStratifiedKFold = None
    MultilabelStratifiedShuffleSplit = None


try:

    from gensim.models import KeyedVectors
except Exception:
    KeyedVectors = None

try:
    import fasttext
    import fasttext.util
except Exception:
    fasttext = None


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
except Exception:
    torch = None


try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception:
    AutoTokenizer = AutoModelForSequenceClassification = TrainingArguments = Trainer = DataCollatorWithPadding = None


import warnings
warnings.filterwarnings("ignore")


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [70]:
def set_seed(seed: int = 42) -> None:
    # Фиксируем seed для воспроизводимости (насколько это возможно).
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    # Настройки для более детерминированного поведения на GPU (если доступно).
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

    # Включаем детерминированные алгоритмы (PyTorch >=1.8). При необходимости окружение CUDA
    # может требовать настройки переменных CUBLAS_WORKSPACE_CONFIG или CPU реализаций.
    # try:
    #     torch.use_deterministic_algorithms(True)
    # except Exception:
    #     try:
    #         torch.set_deterministic(True)
    #     except Exception:
    #         pass

    # Сделаем генератор для DataLoader доступным глобально, чтобы все загрузчики были детерминированы
    global DATA_LOADER_GEN
    try:
        DATA_LOADER_GEN = torch.Generator()
        DATA_LOADER_GEN.manual_seed(seed)
    except Exception:
        DATA_LOADER_GEN = None

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [71]:


# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.tsv', 'validation': 'dev.tsv'}
df_train = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["train"], sep="\t")
df_val = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["validation"], sep="\t")

In [72]:
df_train.head()

,ru_toxic_comment,ru_neutral_comment
0,"и,чё,блядь где этот херой был до этого со свои...","Ну и где этот герой был,со своими доказательст..."
1,"и,чё,блядь где этот херой был до этого со свои...",Где этот герой был до этого со своими доказате...
2,"и,чё,блядь где этот херой был до этого со свои...","и,где этот герой был до этого со своими доказа..."
3,"О, а есть деанон этого петуха?","О, а есть деанон"
4,"херну всякую пишут,из-за этого лайка.долбоебизм.","Чушь всякую пишут, из- за этого лайка."


In [73]:
df_val.head()

,ru_toxic_comment,ru_neutral_comment
0,пиздеж! температуры горения хватит чтобы её ра...,Враньё! Температуры горения хватит чтобы ее ра...
1,пиздеж! температуры горения хватит чтобы её ра...,"неправда,температуры горения хватит чтобы расп..."
2,пиздеж! температуры горения хватит чтобы её ра...,Враньё! Температуры горения хватит на чтобы её...
3,а ты чмо там был.ты вообще служил.гандон,А ты там был? Ты вообще служил?
4,пиздабол ---- а сам где кормишься ?,а сам где кормишься ?


In [74]:
df_train.shape

(11090, 2)

In [75]:
df_val.shape

(1116, 2)

In [76]:
df_train = df_train.drop_duplicates().reset_index(drop=True)
df_val = df_val.drop_duplicates().reset_index(drop=True)

mask_error_train = df_train.astype(str).apply(lambda col: col.str.contains(r"#ERROR!", na=False)).any(axis=1)
mask_error_val = df_val.astype(str).apply(lambda col: col.str.contains(r"#ERROR!", na=False)).any(axis=1)

df_train = df_train.loc[~mask_error_train].reset_index(drop=True)
df_val = df_val.loc[~mask_error_val].reset_index(drop=True)


In [77]:
df_train.shape

(10882, 2)

In [78]:
df_val.shape

(1087, 2)

In [79]:
df_train = df_train.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_val = df_val.sample(frac=1.0, random_state=42).reset_index(drop=True)


In [80]:
df_train, df_test = train_test_split(df_train, test_size=0.1, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [81]:
max_len = 100
vocab_size = 20000
special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

def tokenize(text: str):
    return re.findall(r"\w+|[^\w\s]", str(text).lower(), re.UNICODE)

all_texts = pd.concat([
    df_train["ru_toxic_comment"].astype(str),
    df_train["ru_neutral_comment"].astype(str)
], ignore_index=True)

counter = Counter()
for text in all_texts:
    counter.update(tokenize(text))

most_common = [w for w, _ in counter.most_common(vocab_size - len(special_tokens))]
idx2word = special_tokens + most_common
word2idx = {w: i for i, w in enumerate(idx2word)}

pad_id = word2idx["<PAD>"]
bos_id = word2idx["<BOS>"]
eos_id = word2idx["<EOS>"]
unk_id = word2idx["<UNK>"]


In [82]:
def encode_text(text, max_len):
    toks = tokenize(text)
    tokens = [word2idx.get(w, unk_id) for w in toks]
    tokens = [bos_id] + tokens[:max_len - 2] + [eos_id]
    if len(tokens) < max_len:
        tokens = tokens + [pad_id] * (max_len - len(tokens))
    return tokens

def make_arrays(df):
    src = np.array([encode_text(t, max_len) for t in df["ru_toxic_comment"].astype(str)])
    tgt = np.array([encode_text(t, max_len) for t in df["ru_neutral_comment"].astype(str)])
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    return src, tgt_in, tgt_out


In [83]:
class Seq2SeqDataset(Dataset):
    def __init__(self, src, tgt_in, tgt_out):
        self.src = torch.LongTensor(src)
        self.tgt_in = torch.LongTensor(tgt_in)
        self.tgt_out = torch.LongTensor(tgt_out)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt_in[idx], self.tgt_out[idx]


In [84]:
X_train_src, X_train_tgt_in, X_train_tgt_out = make_arrays(df_train)
X_val_src, X_val_tgt_in, X_val_tgt_out = make_arrays(df_val)
X_test_src, X_test_tgt_in, X_test_tgt_out = make_arrays(df_test)

# Diagnostics: how many UNK in targets
def unk_stats(arr, name):
    total = arr.size
    unk_count = (arr == unk_id).sum()
    print(f"{name}: total tokens={total}, UNK tokens={unk_count}, UNK%={unk_count/total*100:.2f}")

unk_stats(X_train_tgt_out, 'train targets')
unk_stats(X_val_tgt_out, 'val targets')


train targets: total tokens=969507, UNK tokens=1147, UNK%=0.12
val targets: total tokens=107613, UNK tokens=1629, UNK%=1.51


In [85]:
class RNNSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.RNN(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        self.decoder = nn.RNN(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.bidirectional = bidirectional
        enc_out_dim = hidden_size * 2 if bidirectional else hidden_size
        self.enc_proj = nn.Linear(enc_out_dim, hidden_size) if bidirectional else nn.Identity()
        self.attn_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def _merge_directions(self, h):
        if not self.bidirectional:
            return h
        h = h.view(self.num_layers, 2, h.size(1), self.hidden_size)
        return h.sum(dim=1)

    def encode(self, src):
        src_emb = self.dropout(self.embedding(src))
        enc_out, h = self.encoder(src_emb)
        enc_out = self.enc_proj(enc_out)
        h = self._merge_directions(h)
        return enc_out, h

    def decode_step(self, token, hidden, enc_out):
        emb = self.dropout(self.embedding(token))
        dec_out, hidden = self.decoder(emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits, hidden

    def forward(self, src, tgt_in):
        enc_out, h = self.encode(src)
        tgt_emb = self.dropout(self.embedding(tgt_in))
        dec_out, _ = self.decoder(tgt_emb, h)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits

class GRUSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.GRU(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        self.decoder = nn.GRU(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.bidirectional = bidirectional
        enc_out_dim = hidden_size * 2 if bidirectional else hidden_size
        self.enc_proj = nn.Linear(enc_out_dim, hidden_size) if bidirectional else nn.Identity()
        self.attn_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def _merge_directions(self, h):
        if not self.bidirectional:
            return h
        h = h.view(self.num_layers, 2, h.size(1), self.hidden_size)
        return h.sum(dim=1)

    def encode(self, src):
        src_emb = self.dropout(self.embedding(src))
        enc_out, h = self.encoder(src_emb)
        enc_out = self.enc_proj(enc_out)
        h = self._merge_directions(h)
        return enc_out, h

    def decode_step(self, token, hidden, enc_out):
        emb = self.dropout(self.embedding(token))
        dec_out, hidden = self.decoder(emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits, hidden

    def forward(self, src, tgt_in):
        enc_out, h = self.encode(src)
        tgt_emb = self.dropout(self.embedding(tgt_in))
        dec_out, _ = self.decoder(tgt_emb, h)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits

class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.LSTM(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                               dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        self.decoder = nn.LSTM(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                               dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.bidirectional = bidirectional
        enc_out_dim = hidden_size * 2 if bidirectional else hidden_size
        self.enc_proj = nn.Linear(enc_out_dim, hidden_size) if bidirectional else nn.Identity()
        self.attn_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def _merge_directions(self, state):
        if not self.bidirectional:
            return state
        h, c = state
        h = h.view(self.num_layers, 2, h.size(1), self.hidden_size).sum(dim=1)
        c = c.view(self.num_layers, 2, c.size(1), self.hidden_size).sum(dim=1)
        return h, c

    def encode(self, src):
        src_emb = self.dropout(self.embedding(src))
        enc_out, (h, c) = self.encoder(src_emb)
        enc_out = self.enc_proj(enc_out)
        h, c = self._merge_directions((h, c))
        return enc_out, (h, c)

    def decode_step(self, token, hidden, enc_out):
        emb = self.dropout(self.embedding(token))
        dec_out, hidden = self.decoder(emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits, hidden

    def forward(self, src, tgt_in):
        enc_out, hidden = self.encode(src)
        tgt_emb = self.dropout(self.embedding(tgt_in))
        dec_out, _ = self.decoder(tgt_emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits


In [86]:
def train_seq2seq(model, train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32, weight_decay=1e-4, patience=3):
    model = model.to(device)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, generator=DATA_LOADER_GEN)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1)
    best_val = float('inf')
    best_state = None
    bad_epochs = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, tgt_in, tgt_out in train_loader:
            src = src.to(device)
            tgt_in = tgt_in.to(device)
            tgt_out = tgt_out.to(device)
            optimizer.zero_grad()
            logits = model(src, tgt_in)
            loss = loss_fn(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for src, tgt_in, tgt_out in val_loader:
                src = src.to(device)
                tgt_in = tgt_in.to(device)
                tgt_out = tgt_out.to(device)
                logits = model(src, tgt_in)
                loss = loss_fn(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
                val_loss += loss.item()
        val_avg = val_loss / len(val_loader)
        print(f"epoch {epoch+1} train_loss {total_loss/len(train_loader):.4f} val_loss {val_avg:.4f}")
        if val_avg < best_val:
            best_val = val_avg
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [87]:
def greedy_decode(model, src, max_len):
    model.eval()
    src = src.to(device)
    enc_out, hidden = model.encode(src)
    decoded = torch.full((src.size(0), 1), bos_id, dtype=torch.long, device=src.device)
    for _ in range(max_len - 1):
        last = decoded[:, -1:].contiguous()
        logits, hidden = model.decode_step(last, hidden, enc_out)
        next_token = logits.argmax(dim=-1)
        decoded = torch.cat([decoded, next_token], dim=1)
    return decoded


In [88]:
def decode_sequences(seqs):
    texts = []
    for seq in seqs:
        words = []
        for idx in seq:
            if idx == eos_id:
                break
            if idx in (pad_id, bos_id):
                continue
            words.append(idx2word[idx] if idx < len(idx2word) else "")
        texts.append(" ".join(words).strip())
    return texts


In [89]:
train_ds = Seq2SeqDataset(X_train_src, X_train_tgt_in, X_train_tgt_out)
val_ds = Seq2SeqDataset(X_val_src, X_val_tgt_in, X_val_tgt_out)


In [90]:
# train_ds[:5]

(tensor([[    1,  7407,   358,   130,    13,     4,    15,    20,    11,  3122,
             11,   956,     6,  4898,     2,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
         [    1,  1408,    13,    40,   154,     5,   995,   359,     5,   301,
              9,   117,     8,  3123,     2,     0,     0,     0,     0,     0,
              0,     0,     0,     0,  

In [91]:
# val_ds[:5]

(tensor([[    1,   156,   101,     3,    88,    10,   473,    10,    83,     6,
              3,  7852,   186,    10,     8,     3,     6,     2,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
         [    1,     3,  1376,  5191,     8,     3,     6,     6,     6,     6,
              6,     2,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,  

In [92]:
model_rnn = train_seq2seq(RNNSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32)



epoch 1 train_loss 7.4181 val_loss 6.5636
epoch 2 train_loss 6.8271 val_loss 6.4210
epoch 3 train_loss 6.6891 val_loss 6.3268
epoch 4 train_loss 6.5485 val_loss 6.2306
epoch 5 train_loss 6.3873 val_loss 6.0546
epoch 6 train_loss 6.1882 val_loss 5.8984
epoch 7 train_loss 6.0266 val_loss 5.7695
epoch 8 train_loss 5.8855 val_loss 5.6787
epoch 9 train_loss 5.7737 val_loss 5.6165
epoch 10 train_loss 5.6785 val_loss 5.5537


In [93]:
model_gru = train_seq2seq(GRUSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32)


epoch 1 train_loss 7.4370 val_loss 6.6123
epoch 2 train_loss 6.8773 val_loss 6.4999
epoch 3 train_loss 6.6389 val_loss 6.2189
epoch 4 train_loss 6.4100 val_loss 6.0538
epoch 5 train_loss 6.2025 val_loss 5.9280
epoch 6 train_loss 6.0314 val_loss 5.7977
epoch 7 train_loss 5.8761 val_loss 5.8319
epoch 8 train_loss 5.7447 val_loss 5.6594
epoch 9 train_loss 5.6327 val_loss 5.6211
epoch 10 train_loss 5.5286 val_loss 5.5431


In [94]:
model_lstm = train_seq2seq(LSTMSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32)

epoch 1 train_loss 7.4449 val_loss 6.5822
epoch 2 train_loss 6.8793 val_loss 6.4231
epoch 3 train_loss 6.7111 val_loss 6.3549
epoch 4 train_loss 6.5800 val_loss 6.2770
epoch 5 train_loss 6.4348 val_loss 6.1346
epoch 6 train_loss 6.2677 val_loss 5.9877
epoch 7 train_loss 6.0848 val_loss 5.8383
epoch 8 train_loss 5.9041 val_loss 5.7407
epoch 9 train_loss 5.7563 val_loss 5.6525
epoch 10 train_loss 5.6260 val_loss 5.5616


In [95]:
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

from tqdm.auto import tqdm

modelname = "cointegrated/rut5-small"
maxlen = 128

batchsize = 8
numepochs = 2
lr = 1e-5
logsteps = 50
seed = 42

torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(modelname, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(modelname).to(device)

def cleandf(df):
    df = df.dropna()
    df = df[df["ru_toxic_comment"].str.len() > 0]
    df = df[df["ru_neutral_comment"].str.len() > 0]
    return df

df_train = cleandf(df_train)
df_val = cleandf(df_val)

class DetoxSeq2SeqDataset(Dataset):
    def __init__(self, df, tokenizer, maxlen):
        self.inputs = tokenizer(
            ["детоксифицируй текст: " + str(x) for x in df["ru_toxic_comment"].tolist()],
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )

        labels = tokenizer(
            df["ru_neutral_comment"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )["input_ids"]

        labels[labels == tokenizer.pad_token_id] = -100
        self.labels = labels

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.inputs.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = DetoxSeq2SeqDataset(df_train, tokenizer, maxlen)
val_ds = DetoxSeq2SeqDataset(df_val, tokenizer, maxlen)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

train_loader = DataLoader(train_ds, batch_size=batchsize, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=batchsize, shuffle=False, collate_fn=collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

num_training_steps = numepochs * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

def eval_loss(model, loader):
    model.eval()
    total = 0.0
    n = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss

            if torch.isnan(loss):
                continue

            total += loss.item()
            n += 1

    return total / max(1, n)

global_step = 0

for epoch in range(numepochs):
    model.train()

    for step, batch in enumerate(tqdm(train_loader, desc=f"train {epoch+1}/{numepochs}"), start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        out = model(**batch)
        loss = out.loss

        if torch.isnan(loss):
            continue

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        global_step += 1

        if global_step % logsteps == 0:
            print(f"step={global_step} loss={loss.item():.4f}")

    val_loss = eval_loss(model, val_loader)
    print(f"epoch={epoch+1} val_loss={val_loss:.4f}")


class DetoxGenDataset(Dataset):
    def __init__(self, df, tokenizer, maxlen):
        self.inputs = tokenizer(
            ["детоксифицируй текст: " + str(x) for x in df["ru_toxic_comment"].tolist()],
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )

    def __len__(self):
        return self.inputs["input_ids"].size(0)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.inputs.items()}

gen_ds = DetoxGenDataset(df_val, tokenizer, maxlen)
gen_loader = DataLoader(gen_ds, batch_size=batchsize, shuffle=False)

model.eval()
preds = []

with torch.no_grad():
    for batch in tqdm(gen_loader, desc="generate"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            num_beams=4,
        )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        preds.extend([x.strip() for x in decoded])

train 1/2:   4%|▍         | 50/1225 [00:31<12:53,  1.52it/s]

step=50 loss=2.6038


train 1/2:   8%|▊         | 100/1225 [01:02<12:34,  1.49it/s]

step=100 loss=2.1339


train 1/2:  12%|█▏        | 150/1225 [01:34<12:10,  1.47it/s]

...

step=2400 loss=1.7341


train 2/2: 100%|██████████| 1225/1225 [12:47<00:00,  1.60it/s]


step=2450 loss=2.9690


epoch=2 val_loss=1.6100


generate: 100%|██████████| 136/136 [05:15<00:00,  2.32s/it]


In [96]:
# model.eval()
#
# examples = []
#
# with torch.no_grad():
#     for i, batch in enumerate(tqdm(gen_loader, desc="generate")):
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#
#         generated = model.generate(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             max_new_tokens=64,
#             num_beams=4,
#         )
#
#         decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
#
#         for j in range(len(decoded)):
#             idx = i * batchsize + j
#             if idx >= len(df_val):
#                 break
#
#             examples.append({
#                 "toxic": df_val.iloc[idx]["ru_toxic_comment"],
#                 "pred": decoded[j].strip(),
#                 "true": df_val.iloc[idx]["ru_neutral_comment"],
#             })
#
# df_show = pd.DataFrame(examples)
#
# df_show.head(20)

generate: 100%|██████████| 136/136 [05:17<00:00,  2.34s/it]


,toxic,pred,true
0,во всех нациях есть и пидарасы и люди.живите р...,Во всех нациях есть и пидарасы и люди.живите р...,Во всех нациях есть плохие и хорошие люди. Жив...
1,подстрекатель родину гад не любиш .....,Подтрекатель родину гад не любиш,Подстрекатель родину не любишь...
2,"Чмони, чмохи и просто пидоры, я вам в очередно...","Я вам в очередной раз напоминаю, что чмоэстро ...","я вам в очередной раз напоминаю, что он проигр..."
3,"а не охерел ли кравчук? морда не треснет, земл...","А не охерел ли кравчук? морда не треснет, земл...","Кравчук слишком много хочет , в том числе и зе..."
4,мне очень обидно и просто стыдно перед всем ми...,мне очень обидно и просто стыдно перед всем ми...,Мне очень обидно и просто стыдно перед всем ми...
5,а у мну вот чё есть 🙈девочки на днюху подарили...,А у мну вот чё есть 🙈девочки на днюху подарили...,"а у меня вот что есть, девочки на день рождени..."
6,в чем новинка? в пластиковом убогом обвесе? бе...,в чем новинка? в пластиковом убогом обвесе? бе...,Что нового? Пластиковый обвес?
7,"Этот вообще пиздец, контент был максимально уб...",Этот контент был максимально убогим.,Контент был максимально плохим.
8,"недобитый фашистик, снять штанишки и выпороть ...","недобитый фашистик, снять штанишки и выпороть,...",Наказать его надо
9,"руся, перестань, природа одарила тебя большой ...","руся, перестань, природа одарила тебя большой ...",руся перестань природа одарила тебя большой по...


In [97]:
val_src_tensor = torch.LongTensor(X_val_src)

def beam_search_decode(model, src, max_len, beam_width=4):
    model.eval()
    device = next(model.parameters()).device
    outputs = []
    with torch.no_grad():
        for i in range(src.size(0)):
            s = src[i:i+1].to(device)
            enc_out, init_hidden = model.encode(s)

            # beam: list of (tokens, hidden, score)
            beams = [([bos_id], init_hidden, 0.0)]
            completed = []
            for _ in range(max_len - 1):
                new_beams = []
                for tokens, hidden, score in beams:
                    last = torch.LongTensor([[tokens[-1]]]).to(device)
                    logits, h_next = model.decode_step(last, hidden, enc_out)
                    logits = logits[:, -1, :]
                    logp = torch.log_softmax(logits, dim=-1).squeeze(0)
                    topk = torch.topk(logp, beam_width)
                    for k in range(beam_width):
                        token = int(topk.indices[k].item())
                        token_score = float(topk.values[k].item())
                        new_tokens = tokens + [token]
                        new_score = score + token_score
                        new_beams.append((new_tokens, h_next, new_score))
                # keep top beams
                new_beams = sorted(new_beams, key=lambda x: x[2], reverse=True)[:beam_width]
                beams = []
                for tokens, hidden, score in new_beams:
                    if tokens[-1] == eos_id:
                        completed.append((tokens, score))
                    else:
                        beams.append((tokens, hidden, score))
                if len(beams) == 0:
                    break
            if len(completed) == 0:
                completed = [(b[0], b[2]) for b in beams]
            best = sorted(completed, key=lambda x: x[1], reverse=True)[0][0]
            outputs.append(best)
    # pad/truncate outputs to same length
    max_out_len = max(len(x) for x in outputs)
    padded = [x + [pad_id] * (max_out_len - len(x)) for x in outputs]
    return torch.LongTensor(padded)

preds_rnn = decode_sequences(beam_search_decode(model_rnn, val_src_tensor, max_len, beam_width=4).numpy())
preds_gru = decode_sequences(beam_search_decode(model_gru, val_src_tensor, max_len, beam_width=4).numpy())
preds_lstm = decode_sequences(beam_search_decode(model_lstm, val_src_tensor, max_len, beam_width=4).numpy())

refs = df_val["ru_neutral_comment"].astype(str).tolist()


In [98]:
try:
    import evaluate
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")
except Exception as e:
    raise ImportError("Install evaluate to compute BLEU/ROUGE/BERTScore") from e


In [99]:
results = []
preds_transformer = preds
refs_for_bleu = [[r] for r in refs]
for name, preds in [("RNN", preds_rnn), ("GRU", preds_gru), ("LSTM", preds_lstm), ("Transformer", preds_transformer)]:
    try:
        bleu_score = bleu.compute(predictions=preds, references=refs_for_bleu)["bleu"]
    except Exception:
        bleu_score = 0.0
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="ru")
    results.append({
        "model": name,
        "bleu": bleu_score,
        "rouge1": rouge_scores.get("rouge1"),
        "rouge2": rouge_scores.get("rouge2"),
        "rougeL": rouge_scores.get("rougeL"),
        "bertscore_f1": float(np.mean(bert_scores["f1"]))
    })

metrics_df = pd.DataFrame(results)
print(metrics_df)


         model      bleu    rouge1   rouge2    rougeL  bertscore_f1
0          RNN  0.023914  0.000000  0.00000  0.000000      0.652625
1          GRU  0.023524  0.000000  0.00000  0.000000      0.640651
2         LSTM  0.018124  0.000000  0.00000  0.000000      0.641806
3  Transformer  0.388288  0.034564  0.00736  0.034301      0.834285


In [100]:
# from collections import Counter
# import numpy as np
#
# def diagnostics(preds, refs, name, n=10):
#     print(name)
#     print("preds len:", len(preds), "refs len:", len(refs))
#     empty = sum(1 for p in preds if not str(p).strip())
#     print("empty preds:", empty)
#     lens_pred = Counter(len(p.split()) for p in preds)
#     lens_ref = Counter(len(r.split()) for r in refs)
#     print("pred token length distribution (top 5):", lens_pred.most_common(5))
#     print("ref token length distribution (top 5):", lens_ref.most_common(5))
#     print("examples:")
#     for i in range(min(n, len(preds))):
#         print(i, "P:", repr(preds[i]), "\n   R:", repr(refs[i]), "\n")
#
# diagnostics(preds_rnn, refs, "RNN")
# diagnostics(preds_gru, refs, "GRU")
# diagnostics(preds_lstm, refs, "LSTM")
# diagnostics(preds, refs, "Transformer (raw preds)")
#
# refs_for_bleu = [[r] for r in refs]
#
# bleu_score_rnn = bleu.compute(predictions=preds_rnn, references=refs_for_bleu)["bleu"]
# bleu_score_gru = bleu.compute(predictions=preds_gru, references=refs_for_bleu)["bleu"]
# bleu_score_lstm = bleu.compute(predictions=preds_lstm, references=refs_for_bleu)["bleu"]
# bleu_score_trans = bleu.compute(predictions=preds, references=refs_for_bleu)["bleu"]
#
# rouge_rnn = rouge.compute(predictions=preds_rnn, references=refs)
# rouge_gru = rouge.compute(predictions=preds_gru, references=refs)
# rouge_lstm = rouge.compute(predictions=preds_lstm, references=refs)
# rouge_trans = rouge.compute(predictions=preds, references=refs)
#
# bert_rnn = bertscore.compute(predictions=preds_rnn, references=refs, lang="ru")
# bert_gru = bertscore.compute(predictions=preds_gru, references=refs, lang="ru")
# bert_lstm = bertscore.compute(predictions=preds_lstm, references=refs, lang="ru")
# bert_trans = bertscore.compute(predictions=preds, references=refs, lang="ru")
#
# results = []
# for name, bleu_s, rouge_s, bert_s in [
#     ("RNN", bleu_score_rnn, rouge_rnn, bert_rnn),
#     ("GRU", bleu_score_gru, rouge_gru, bert_gru),
#     ("LSTM", bleu_score_lstm, rouge_lstm, bert_lstm),
#     ("Transformer", bleu_score_trans, rouge_trans, bert_trans),
# ]:
#     results.append({
#         "model": name,
#         "bleu": float(bleu_s),
#         "rouge1": rouge_s.get("rouge1"),
#         "rouge2": rouge_s.get("rouge2"),
#         "rougeL": rouge_s.get("rougeL"),
#         "bertscore_f1": float(np.mean(bert_s["f1"]))
#     })
#
# metrics_df = pd.DataFrame(results)
# print(metrics_df)

RNN
preds len: 1087 refs len: 1087
empty preds: 0
pred token length distribution (top 5): [(4, 158), (5, 114), (6, 107), (8, 103), (7, 97)]
ref token length distribution (top 5): [(5, 135), (6, 112), (7, 99), (4, 98), (9, 92)]
examples:
0 P: 'пусть , надо и <UNK> и теперь и все .' 
   R: 'Во всех нациях есть плохие и хорошие люди. Живите радуйтесь жизни и не болейте' 

1 P: 'не хочу не могу . . .' 
   R: 'Подстрекатель родину не любишь...' 

2 P: 'зачем и вы , я бы в тобой , я не могу .' 
   R: 'я вам в очередной раз напоминаю, что он проиграл суд первой инстанции Олеже Ссаколову' 

3 P: 'а не хочу , что не могу , <UNK> и <UNK>' 
   R: 'Кравчук слишком много хочет , в том числе и земли , такие , как он не во что не ставят людей .' 

4 P: 'мне хочу <UNK> и вы , что у меня <UNK> что у меня <UNK>' 
   R: 'Мне очень обидно и просто стыдно перед всем миром, что у нас такое ужасное правительство. Как такие люди попадают в правительство?' 

5 P: 'а у <UNK> , что по тобой на тобой .' 
   R: 'а